# Rashi Harness Sovereign Cloud Runner + Autonomous Browser Worker
This notebook runs the backend cloud compute engine for **Rashi Harness**:
1. **Pre-configured API Keys**: Respan, Explabs, APInex, and Coarena.
2. **Automated Browser Engine**: Installs and verifies Headless Chromium + Playwright for web recon and automation.
3. **Cloudflare Tunnel**: Exposes an encrypted endpoint back to your local Rashi Harness desktop app.

In [ ]:
# Step 1: Install Dependencies & Cloud Browser
!pip install -q fastapi uvicorn pycloudflared pydantic playwright
!playwright install --with-deps chromium

In [ ]:
# Step 2: Launch Rashi Cloud Server with Browser Automation & Tunnel
import os, subprocess, threading, base64
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
import uvicorn
from pycloudflared import try_cloudflare

os.environ['RESPAN_API_KEY'] = '04EkULNO.9rBUnYLdYCNCdDtyO2RtOKzN4pY9XTlG'
os.environ['EXPLABS_API_KEY'] = 'xpl_35be619b5271cdfc433015f3d536227d86a2bdbb'
os.environ['APINEX_API_KEY'] = 'sk-apx3055d1309381796de30bbbdf3ef93bd4e82c349040c0bfe'
os.environ['COARENA_API_KEY'] = 'coa_dc2ff39b6065c5c4244dfd1fd84ff1c0'

app = FastAPI(title='Rashi Harness Cloud Engine')

@app.get('/health')
def health():
    return {'status': 'online', 'browser_ready': True, 'engine': 'Rashi Cloud'}

class BrowserNavRequest(BaseModel):
    url: str
    take_screenshot: bool = True

@app.post('/browser/navigate')
async def browser_navigate(req: BrowserNavRequest):
    from playwright.async_api import async_playwright
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        page = await browser.new_page()
        await page.goto(req.url, timeout=30000)
        title = await page.title()
        b64 = None
        if req.take_screenshot:
            img_bytes = await page.screenshot(full_page=False)
            b64 = base64.b64encode(img_bytes).decode('utf-8')
        await browser.close()
        return {'url': req.url, 'title': title, 'screenshot_base64': b64}

def run():
    uvicorn.run(app, host='0.0.0.0', port=8000)

threading.Thread(target=run, daemon=True).start()
tunnel = try_cloudflare(port=8000)
print('\n' + '='*60)
print(f'RASHI CLOUD WORKER LIVE AT: {tunnel.tunnel}')
print(f'To link locally run: colab.bat --add-endpoint {tunnel.tunnel}')
print('='*60)